# Aggregate Results

Downloads all solver result datasets, merges them, computes gaps, Wilcoxon tests with Bonferroni correction, generates plots, and exports final summary.

In [ ]:
import os, subprocess, sys, json, csv, glob
import numpy as np

subprocess.run(["pip", "install", "-q", "numpy", "scipy", "matplotlib", "pandas", "kagglehub", "requests"], check=True)

INSTANCE_DIR = "/kaggle/working/instances"
if not os.path.exists(INSTANCE_DIR):
    try:
        import kagglehub
        path = kagglehub.dataset_download("elfateh/dana-instances")
        subprocess.run(["cp", "-r", os.path.join(path, "*"), "/kaggle/working/instances/"], check=True)
    except Exception as e:
        print(f"Kaggle download failed, trying GitHub: {e}")
        import requests as _requests
        BASE_URL = "https://raw.githubusercontent.com/PyVRP/Instances/main"
        SETS = {
            "cordeau": ("MDVRPTW", [f"PR{i}{s}.vrp" for i in range(11, 25) for s in ("A", "B")]),
            "solomon": ("VRPTW/Solomon", ["C101.vrp","C102.vrp","C103.vrp","C104.vrp","C105.vrp","C106.vrp","C107.vrp","C108.vrp","C109.vrp","C201.vrp","C202.vrp","C203.vrp","C204.vrp","C205.vrp","C206.vrp","C207.vrp","C208.vrp","R101.vrp","R102.vrp","R103.vrp","R104.vrp","R105.vrp","R106.vrp","R107.vrp","R108.vrp","R109.vrp","R110.vrp","R111.vrp","R112.vrp","R201.vrp","R202.vrp","R203.vrp","R204.vrp","R205.vrp","R206.vrp","R207.vrp","R208.vrp","R209.vrp","R210.vrp","R211.vrp","RC101.vrp","RC102.vrp","RC103.vrp","RC104.vrp","RC105.vrp","RC106.vrp","RC107.vrp","RC108.vrp","RC201.vrp","RC202.vrp","RC203.vrp","RC204.vrp","RC205.vrp","RC206.vrp","RC207.vrp","RC208.vrp"]),
            "gehring": ("VRPTW", [f"{t}{n}_10_{i}.vrp" for t in ("C","R","RC") for n in (1,2) for i in range(1,11)]),
            "x_instances": ("CVRP", ["X-n101-k25.vrp","X-n106-k14.vrp","X-n110-k13.vrp","X-n115-k10.vrp","X-n120-k6.vrp","X-n125-k30.vrp","X-n129-k18.vrp","X-n134-k13.vrp","X-n139-k10.vrp","X-n143-k7.vrp","X-n153-k22.vrp","X-n157-k13.vrp","X-n162-k11.vrp","X-n167-k10.vrp","X-n176-k26.vrp","X-n181-k23.vrp","X-n186-k15.vrp","X-n190-k8.vrp","X-n195-k51.vrp","X-n200-k36.vrp","X-n204-k19.vrp","X-n209-k16.vrp","X-n214-k11.vrp","X-n219-k73.vrp","X-n223-k34.vrp","X-n228-k23.vrp","X-n233-k16.vrp","X-n237-k14.vrp","X-n242-k48.vrp","X-n247-k50.vrp"]),
        }
        for sname, (rdir, files) in SETS.items():
            dest = os.path.join(INSTANCE_DIR, sname)
            os.makedirs(dest, exist_ok=True)
            for fname in files:
                url = f"{BASE_URL}/{rdir}/{fname}"
                path_d = os.path.join(dest, fname)
                if not os.path.exists(path_d):
                    try:
                        r = _requests.get(url, timeout=30)
                        if r.status_code == 200:
                            with open(path_d, "w") as f:
                                f.write(r.text)
                    except Exception:
                        pass
        print("Downloaded instances from GitHub")


In [ ]:
# Download all result datasets
SOLVERS = ["dana", "pyvrp", "hgs", "lkh3", "ortools", "pomo", "am", "bq-nco", "goal", "deepaco", "parco", "rrnco"]
RESULT_DIR = "/kaggle/working/results"
os.makedirs(RESULT_DIR, exist_ok=True)

for solver in SOLVERS:
    dest = os.path.join(RESULT_DIR, f"{solver}_results.csv")
    if os.path.exists(dest):
        continue
    try:
        import kagglehub
        path = kagglehub.dataset_download(f"elfateh/dana-results-{solver}")
        csvs = glob.glob(os.path.join(path, "*.csv"))
        if csvs:
            subprocess.run(["cp", csvs[0], dest], check=True)
            print(f"  {solver}: downloaded.")
        else:
            print(f"  {solver}: no CSV found in dataset.")
    except Exception as e:
        print(f"  {solver}: download failed: {e}")

# Merge all CSVs
import pandas as pd

all_dfs = []
for f in sorted(glob.glob(os.path.join(RESULT_DIR, "*_results.csv"))):
    try:
        df = pd.read_csv(f)
        all_dfs.append(df)
    except Exception as e:
        print(f"  Error reading {f}: {e}")

if all_dfs:
    merged = pd.concat(all_dfs, ignore_index=True)
    merged.to_csv(os.path.join(RESULT_DIR, "all_results.csv"), index=False)
    print(f"\nMerged {len(merged)} rows from {len(all_dfs)} files.")
else:
    print("No results to merge.")
    merged = pd.DataFrame()

In [ ]:
# Compute gaps: reference = best raw_euclidean_cost per instance across all solvers
def compute_gap(cost, best_known):
    return 100.0 * (cost - best_known) / best_known


if not merged.empty:
    # Find best cost per instance
    best_per_instance = merged.groupby("instance")["raw_euclidean_cost"].min()
    
    merged["best_known"] = merged["instance"].map(best_per_instance)
    merged["gap"] = merged.apply(
        lambda row: compute_gap(row["raw_euclidean_cost"], row["best_known"])
        if pd.notna(row["raw_euclidean_cost"]) and pd.notna(row["best_known"]) else None,
        axis=1,
    )
    
    # Print summary per solver
    print("\n=== Solver Summary (raw_euclidean_cost) ===")
    for solver in sorted(merged["solver"].unique()):
        sdf = merged[merged["solver"] == solver].dropna(subset=["raw_euclidean_cost"])
        if sdf.empty:
            print(f"  {solver}: no results")
            continue
        gaps = sdf["gap"].dropna()
        print(f"  {solver}: n={len(sdf)}, mean_cost={sdf['raw_euclidean_cost'].mean():.2f}, "
              f"mean_gap={gaps.mean():.2f}% (+/-{gaps.std():.2f}%), "
              f"min_gap={gaps.min():.2f}%, max_gap={gaps.max():.2f}%")

    merged.to_csv(os.path.join(RESULT_DIR, "all_results_with_gaps.csv"), index=False)
    print(f"\nSaved all_results_with_gaps.csv")

In [ ]:
# Wilcoxon signed-rank tests with Bonferroni correction
from scipy.stats import wilcoxon


if not merged.empty:
    # Reference solver = pyvrp (or best available)
    ref_solver = "pyvrp" if "pyvrp" in merged["solver"].values else merged["solver"].value_counts().index[0]
    ref_df = merged[merged["solver"] == ref_solver].dropna(subset=["raw_euclidean_cost"])
    ref_costs = ref_df.set_index("instance")["raw_euclidean_cost"]
    
    print(f"\n=== Wilcoxon Tests (reference: {ref_solver}) ===")
    comparisons = {}
    for solver in sorted(merged["solver"].unique()):
        if solver == ref_solver:
            continue
        sdf = merged[merged["solver"] == solver].dropna(subset=["raw_euclidean_cost"])
        if sdf.empty:
            continue
        sol_costs = sdf.set_index("instance")["raw_euclidean_cost"]
        common = ref_costs.index.intersection(sol_costs.index)
        if len(common) < 6:
            print(f"  {solver}: n={len(common)} < 6, skipping test")
            continue
        ref_vals = ref_costs[common].values.astype(float)
        sol_vals = sol_costs[common].values.astype(float)
        try:
            stat, p = wilcoxon(ref_vals, sol_vals, alternative="less")
            comparisons[solver] = {"p_value": float(p), "n": len(common)}
            print(f"  {solver}: p={p:.6f} (n={len(common)})")
        except Exception as e:
            print(f"  {solver}: test failed: {e}")
    
    # Bonferroni correction
    if comparisons:
        raw_p = [c["p_value"] for c in comparisons.values()]
        m = len(raw_p)
        corrected = [min(p * m, 1.0) for p in raw_p]
        for solver, p_corr in zip(comparisons.keys(), corrected):
            comparisons[solver]["p_value_corrected"] = p_corr
            comparisons[solver]["significant"] = p_corr < 0.05
            print(f"  {solver} (Bonferroni): p_corr={p_corr:.6f}, sig={p_corr < 0.05}")

In [ ]:
# Generate plots
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

if not merged.empty:
    # Box plot of gaps per solver
    fig, ax = plt.subplots(figsize=(12, 6))
    solver_gaps = {}
    for solver in sorted(merged["solver"].unique()):
        sdf = merged[merged["solver"] == solver].dropna(subset=["gap"])
        if not sdf.empty:
            solver_gaps[solver] = sdf["gap"].values
    
    if solver_gaps:
        positions = np.arange(len(solver_gaps))
        bp = ax.boxplot(list(solver_gaps.values()), positions=positions, patch_artist=True)
        colors = plt.cm.tab10(np.linspace(0, 1, len(solver_gaps)))
        for patch, color in zip(bp["boxes"], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
        ax.set_xticklabels(list(solver_gaps.keys()), rotation=45, ha="right")
        ax.set_ylabel("Gap vs Best Known (%)")
        ax.set_title("Gap Distribution Across Solvers")
        ax.grid(True, alpha=0.3, axis="y")
        ax.axhline(y=0, color="black", linestyle="--", alpha=0.5)
        plt.tight_layout()
        plt.savefig(os.path.join(RESULT_DIR, "gap_distribution.png"), dpi=150, bbox_inches="tight")
        plt.close()
        print("Saved gap_distribution.png")

    # Scatter: cost vs solver per benchmark
    for bench in merged["benchmark"].unique():
        fig, ax = plt.subplots(figsize=(10, 5))
        bench_df = merged[(merged["benchmark"] == bench) & merged["raw_euclidean_cost"].notna()]
        for solver in sorted(bench_df["solver"].unique()):
            sdf = bench_df[bench_df["solver"] == solver]
            ax.scatter(range(len(sdf)), sdf["raw_euclidean_cost"], label=solver, s=30, alpha=0.7)
        ax.set_xlabel("Instance Index")
        ax.set_ylabel("Raw Euclidean Cost")
        ax.set_title(f"{bench}: Cost by Solver")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(RESULT_DIR, f"{bench}_costs.png"), dpi=150, bbox_inches="tight")
        plt.close()
        print(f"Saved {bench}_costs.png")

In [ ]:
# Export final summary JSON
if not merged.empty:
    summary = {}
    for solver in sorted(merged["solver"].unique()):
        sdf = merged[merged["solver"] == solver].dropna(subset=["raw_euclidean_cost"])
        if sdf.empty:
            continue
        gaps = sdf["gap"].dropna()
        summary[solver] = {
            "n": len(sdf),
            "mean_cost": float(sdf["raw_euclidean_cost"].mean()),
            "std_cost": float(sdf["raw_euclidean_cost"].std()),
            "mean_gap": float(gaps.mean()) if len(gaps) > 0 else None,
            "std_gap": float(gaps.std()) if len(gaps) > 0 else None,
            "min_gap": float(gaps.min()) if len(gaps) > 0 else None,
            "max_gap": float(gaps.max()) if len(gaps) > 0 else None,
        }
    
    with open(os.path.join(RESULT_DIR, "summary.json"), "w") as f:
        json.dump(summary, f, indent=2)
    print("Saved summary.json")
    
    # Print final comparison table
    print("\n=== Final Comparison ===")
    print(f"{'Solver':<15} {'N':<5} {'Mean Cost':<12} {'Mean Gap':<12} {'Min Gap':<10} {'Max Gap':<10}")
    print("-" * 65)
    for solver, data in sorted(summary.items()):
        print(f"{solver:<15} {data['n']:<5} {data['mean_cost']:<12.2f} "
              f"{data['mean_gap']:<12.2f} {data['min_gap']:<10.2f} {data['max_gap']:<10.2f}")

In [ ]:
# Upload final aggregated results
DATASET_DIR = "/kaggle/working/dana-benchmark-results"
os.makedirs(DATASET_DIR, exist_ok=True)
for f in glob.glob(os.path.join(RESULT_DIR, "*.csv")) + glob.glob(os.path.join(RESULT_DIR, "*.json")) + glob.glob(os.path.join(RESULT_DIR, "*.png")):
    subprocess.run(["cp", f, DATASET_DIR], check=True)

with open(os.path.join(DATASET_DIR, "dataset-metadata.json"), "w") as f:
    json.dump({
        "title": "dana-benchmark-results",
        "id": "elfateh/dana-benchmark-results",
        "licenses": [{"name": "CC0-1.0"}],
    }, f)

# Always try version first (dataset already exists), fall back to create
r = subprocess.run(
    ["kaggle", "datasets", "version", "-p", DATASET_DIR, "-m", "Updated results", "--dir-mode", "zip"],
    capture_output=True, text=True,
)
if r.returncode != 0:
    r2 = subprocess.run(
        ["kaggle", "datasets", "create", "-p", DATASET_DIR, "--dir-mode", "zip"],
        capture_output=True, text=True,
    )
    if r2.returncode != 0:
        print(f"Dataset upload failed: {r2.stderr.strip()}")
    else:
        print("Final results dataset created successfully.")
else:
    print("Final results dataset version updated successfully.")